# Baseline notebook: churn early-warning (Task 02)
**Purpose:** show what a *non-ML* baseline and a simple model achieve on 12 labelled customers, with honest uncertainty.
**Warning:** n = 12 (5 churners). Every number below is illustrative, not evidence of production performance.

In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, brier_score_loss, roc_auc_score
df = pd.read_csv('customer-churn-training.csv')
print(df.shape); print(df.isna().sum().sum(), 'missing cells;', df.duplicated().sum(), 'duplicate rows;', df.customer_id.duplicated().sum(), 'duplicate IDs')
print(df.churned.value_counts().rename({0:'stayed',1:'churned'}).to_string())
print(df.groupby('churned').mean(numeric_only=True).round(1).to_string())

(12, 7)
0 missing cells; 0 duplicate rows; 0 duplicate IDs
churned
stayed     7
churned    5
         tenure_months  support_tickets  monthly_spend_inr  last_login_days
churned                                                                    
0                 17.9              1.1             1127.6              3.7
1                  4.2              3.8              619.0             18.8


## Quality checks
No missing values or duplicates. The class balance is 7:5 (41.7% churn). Check separation per feature below.

In [2]:
for c in ['last_login_days','support_tickets','tenure_months','monthly_spend_inr']:
    print(c, '| churners:', sorted(df[df.churned==1][c]), '| stayers:', sorted(df[df.churned==0][c]))
print(pd.crosstab(df.plan_type, df.churned))

last_login_days | churners: [12, 14, 17, 21, 30] | stayers: [1, 2, 2, 3, 4, 5, 9]
support_tickets | churners: [3, 3, 4, 4, 5] | stayers: [0, 0, 1, 1, 2, 2, 2]
tenure_months | churners: [1, 2, 4, 6, 8] | stayers: [7, 11, 15, 18, 20, 24, 30]
monthly_spend_inr | churners: [499, 499, 499, 799, 799] | stayers: [799, 999, 999, 999, 1299, 1299, 1499]
churned    0  1
plan_type      
Basic      0  5
Pro        3  0
Standard   4  0


**Finding:** three signals separate the classes perfectly on these 12 rows: `last_login_days` (stayers <= 9, churners >= 12), `support_tickets` (stayers <= 2, churners >= 3) and `plan_type` (all 5 churners are Basic; no Pro or Standard customer churned). Perfect separation on 12 rows is a warning, not good news: it may be chance, a synthetic/toy dataset, or leakage (inactivity and tickets may be recorded *after* the customer decided to leave). Snapshot timing must be confirmed before any feature is trusted.

## Non-ML baselines (leave-one-out evaluation)

In [3]:
y = df.churned.values
def report(name, pred, proba=None):
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    s = f'{name:34s} TP={tp} FP={fp} FN={fn} TN={tn} precision={precision_score(y,pred,zero_division=0):.2f} recall={recall_score(y,pred):.2f}'
    if proba is not None: s += f' brier={brier_score_loss(y,proba):.3f} auc={roc_auc_score(y,proba):.2f}'
    print(s)
report('Always predict "stay"', np.zeros(12,int))
report('Rule: login gap >= 10 days', (df.last_login_days>=10).astype(int))
report('Rule: tenure <= 6 months', (df.tenure_months<=6).astype(int))
report('Rule: >=3 support tickets', (df.support_tickets>=3).astype(int))

Always predict "stay"              TP=0 FP=0 FN=5 TN=7 precision=0.00 recall=0.00
Rule: login gap >= 10 days         TP=5 FP=0 FN=0 TN=7 precision=1.00 recall=1.00
Rule: tenure <= 6 months           TP=4 FP=0 FN=1 TN=7 precision=1.00 recall=0.80
Rule: >=3 support tickets          TP=5 FP=0 FN=0 TN=7 precision=1.00 recall=1.00


Note: the login rule is chosen by looking at all 12 rows, so its perfect score is **optimistic** (threshold fitted on the evaluation data). The majority baseline has zero recall, so any useful model must beat the rules, not just 'stay'.

## Simple ML model, leave-one-out

In [4]:
num = ['tenure_months','support_tickets','monthly_spend_inr','last_login_days']
def loo(features, C=0.5):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    return cross_val_predict(m, df[features], y, cv=LeaveOneOut(), method='predict_proba')[:,1]
for name,f in [('All 4 numeric', num), ('Without last_login_days', [c for c in num if c!='last_login_days'])]:
    p = loo(f); report('LogReg '+name, (p>=0.5).astype(int), p)

LogReg All 4 numeric               TP=5 FP=1 FN=0 TN=6 precision=0.83 recall=1.00 brier=0.073 auc=1.00
LogReg Without last_login_days     TP=5 FP=1 FN=0 TN=6 precision=0.83 recall=1.00 brier=0.086 auc=0.97


**Reading the result:** the model without the suspect login feature is the realistic one to compare with. Differences of one or two customers are noise at this sample size. Calibration cannot be assessed with 12 points; the Brier score is reported only for completeness.

In [5]:
# Cost-aware threshold: assumed costs (INR per customer), see memo. FN = lost customer margin; FP = wasted offer.
COST_FN, COST_FP = 6000, 400
p = loo([c for c in num if c!='last_login_days'])
for t in [0.3,0.4,0.5,0.6]:
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    print(f'threshold {t}: FN={fn} FP={fp} expected cost = INR {fn*COST_FN+fp*COST_FP}')
print('Cost if no action at all: INR', int(y.sum()*COST_FN))

threshold 0.3: FN=0 FP=2 expected cost = INR 800
threshold 0.4: FN=0 FP=1 expected cost = INR 400
threshold 0.5: FN=0 FP=1 expected cost = INR 400
threshold 0.6: FN=1 FP=0 expected cost = INR 6000
Cost if no action at all: INR 30000


## Conclusion
1. With 12 rows, no model is trustworthy; the notebook only proves the pipeline and the evaluation protocol.
2. A transparent inactivity rule is the baseline any ML model must beat on a larger, time-split dataset.
3. Proceed to ML only after (a) leakage review of `last_login_days` and `support_tickets`, (b) 1,000+ labelled customers with >=100 churn events, (c) a time-based train/test split.